# Week 12 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 12 Day 01: Momentum strategy design

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Evaluate two classic strategy families under unified, realistic testing conditions.

## Continuity and Handoff
- Previous checkpoint: Week 11 Day 07: Portfolio Project
- Previous lesson file: content/week-11/day-07.md
- Today's deliverable: Implement a momentum signal function with parameter control.
- Next handoff target: Week 12 Day 02: Mean reversion strategy design
- Next lesson file: content/week-12/day-02.md

## Theory Concepts

### Concept 1: Trend persistence hypothesis
Trend persistence hypothesis is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Lookback and holding period tradeoffs
Lookback and holding period tradeoffs is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Signal smoothing
Signal smoothing is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
**Plain-English interpretation:** Removes non-stationary level drift.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 2: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
**Plain-English interpretation:** Lag-memory measurement.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 3: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
**Plain-English interpretation:** One-step dependence model.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\phi$ | Autoregressive coefficient | dimensionless | 0.64 |
| $e_t$ | Forecast residual | return units | -0.003 |
| $\lambda$ | EWMA decay factor | 0 to 1 | 0.94 |

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Build a cross-asset momentum score and rank signals.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Momentum strategy design'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement a momentum signal function with parameter control.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 12 Day 01): Explain First Difference and define every symbol clearly.
   - Model answer: "I use First Difference to convert raw prices into a decision-ready metric. The formula is $\Delta x_t=x_t-x_{t-1}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Momentum strategy design' matter in live trading systems?
   - Model answer: "Momentum strategy design matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Momentum strategy design as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
Which momentum parameter is most sensitive to regime changes?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 12 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1201)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


{'ar1_phi': -0.10249606319354143,
 'next_return_forecast': -0.0012387351468446841,
 'acf_lag1': -0.1024823418192947,
 'acf_lag5': -0.0004522448476444111,
 'naive_rmse': 0.016055259104041385}

## Week 12 Day 01 Quiz

Topic: **Momentum strategy design**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [3]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 113.000
price_t = 113.960
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Momentum strategy design')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.008496)
print('  gross_return_expected  =', 1.008496)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 113.0
  P_t    : 113.96
  r_t    : 0.008496 => 0.85%
  1+r_t  : 1.008496

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Momentum strategy design
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.008496
  gross_return_expected  = 1.008496


# Week 12 Day 02: Mean reversion strategy design

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Evaluate two classic strategy families under unified, realistic testing conditions.

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 01: Momentum strategy design
- Previous lesson file: content/week-12/day-01.md
- Today's deliverable: Implement a mean-reversion signal with adaptive bands.
- Next handoff target: Week 12 Day 03: Unified backtest comparison
- Next lesson file: content/week-12/day-03.md

## Theory Concepts

### Concept 1: Reversion-to-mean hypothesis
Reversion-to-mean hypothesis is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Z-score normalization
Z-score normalization is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Entry/exit band selection
Entry/exit band selection is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
**Plain-English interpretation:** Lag-memory measurement.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 2: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
**Plain-English interpretation:** One-step dependence model.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 3: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
**Plain-English interpretation:** Adaptive volatility estimate.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\phi$ | Autoregressive coefficient | dimensionless | 0.64 |
| $e_t$ | Forecast residual | return units | -0.003 |
| $\lambda$ | EWMA decay factor | 0 to 1 | 0.94 |

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Construct a z-score reversion signal on spread-like data.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Mean reversion strategy design'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement a mean-reversion signal with adaptive bands.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 12 Day 02): Explain Autocorrelation and define every symbol clearly.
   - Model answer: "I use Autocorrelation to convert raw prices into a decision-ready metric. The formula is $\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Mean reversion strategy design' matter in live trading systems?
   - Model answer: "Mean reversion strategy design matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Mean reversion strategy design as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
How do you detect when mean reversion has structurally broken?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 12 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [4]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1202)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


{'ar1_phi': -0.10249592813047463,
 'next_return_forecast': -0.001238733514514986,
 'acf_lag1': -0.10248220685064457,
 'acf_lag5': -0.0004519504034786784,
 'naive_rmse': 0.016055256056851906}

## Week 12 Day 02 Quiz

Topic: **Mean reversion strategy design**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [5]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 114.000
price_t = 115.026
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Mean reversion strategy design')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 114.0
  P_t    : 115.026
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Mean reversion strategy design
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009


# Week 12 Day 03: Unified backtest comparison

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Evaluate two classic strategy families under unified, realistic testing conditions.

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 02: Mean reversion strategy design
- Previous lesson file: content/week-12/day-02.md
- Today's deliverable: Create side-by-side strategy comparison table.
- Next handoff target: Week 12 Day 04: Robustness and sensitivity analysis
- Next lesson file: content/week-12/day-04.md

## Theory Concepts

### Concept 1: Comparable assumptions
Comparable assumptions is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Cost-normalized evaluation
Cost-normalized evaluation is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Risk parity of strategy variants
Risk parity of strategy variants is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
**Plain-English interpretation:** One-step dependence model.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 2: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
**Plain-English interpretation:** Adaptive volatility estimate.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 3: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
**Plain-English interpretation:** Forecast error benchmark.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\phi$ | Autoregressive coefficient | dimensionless | 0.64 |
| $e_t$ | Forecast residual | return units | -0.003 |
| $\lambda$ | EWMA decay factor | 0 to 1 | 0.94 |

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Run both strategies under identical data and cost assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Unified backtest comparison'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create side-by-side strategy comparison table.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 12 Day 03): Explain AR(1) and define every symbol clearly.
   - Model answer: "I use AR(1) to convert raw prices into a decision-ready metric. The formula is $x_t=c+\phi x_{t-1}+\epsilon_t$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Unified backtest comparison' matter in live trading systems?
   - Model answer: "Unified backtest comparison matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Unified backtest comparison as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
Is any performance gap just a risk-budget mismatch?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 12 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [6]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1203)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


{'ar1_phi': -0.10249592813047463,
 'next_return_forecast': -0.001238733514514986,
 'acf_lag1': -0.10248220685064457,
 'acf_lag5': -0.0004519504034786784,
 'naive_rmse': 0.016055256056851906}

## Week 12 Day 03 Quiz

Topic: **Unified backtest comparison**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [7]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 115.000
price_t = 116.093
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Unified backtest comparison')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009504)
print('  gross_return_expected  =', 1.009504)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 115.0
  P_t    : 116.093
  r_t    : 0.009504 => 0.95%
  1+r_t  : 1.009504

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Unified backtest comparison
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.009504
  gross_return_expected  = 1.009504


# Week 12 Day 04: Robustness and sensitivity analysis

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Evaluate two classic strategy families under unified, realistic testing conditions.

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 03: Unified backtest comparison
- Previous lesson file: content/week-12/day-03.md
- Today's deliverable: Implement parameter grid with robustness summary outputs.
- Next handoff target: Week 12 Day 05: Capstone narrative and decision
- Next lesson file: content/week-12/day-05.md

## Theory Concepts

### Concept 1: Parameter sweeps
Parameter sweeps is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Sub-period stability
Sub-period stability is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Stress scenarios
Stress scenarios is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
**Plain-English interpretation:** Adaptive volatility estimate.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 2: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
**Plain-English interpretation:** Forecast error benchmark.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 3: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
**Plain-English interpretation:** Removes non-stationary level drift.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\phi$ | Autoregressive coefficient | dimensionless | 0.64 |
| $e_t$ | Forecast residual | return units | -0.003 |
| $\lambda$ | EWMA decay factor | 0 to 1 | 0.94 |

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Evaluate strategy sensitivity across lookback and threshold ranges.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Robustness and sensitivity analysis'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement parameter grid with robustness summary outputs.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 12 Day 04): Explain EWMA Vol and define every symbol clearly.
   - Model answer: "I use EWMA Vol to convert raw prices into a decision-ready metric. The formula is $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Robustness and sensitivity analysis' matter in live trading systems?
   - Model answer: "Robustness and sensitivity analysis matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Robustness and sensitivity analysis as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
Which strategy is more robust after stress tests?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 12 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1204)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


{'ar1_phi': -0.10249596439461026,
 'next_return_forecast': -0.0012387339527919108,
 'acf_lag1': -0.10248224286555785,
 'acf_lag5': -0.00045265569942450735,
 'naive_rmse': 0.016055253500882848}

## Week 12 Day 04 Quiz

Topic: **Robustness and sensitivity analysis**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [9]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 116.000
price_t = 117.160
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Robustness and sensitivity analysis')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 116.0
  P_t    : 117.16
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Robustness and sensitivity analysis
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01


# Week 12 Day 05: Capstone narrative and decision

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Evaluate two classic strategy families under unified, realistic testing conditions.

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 04: Robustness and sensitivity analysis
- Previous lesson file: content/week-12/day-04.md
- Today's deliverable: Generate final capstone report artifact with plots and metrics.
- Next handoff target: Week 12 Day 06: Revision Sprint
- Next lesson file: content/week-12/day-06.md

## Theory Concepts

### Concept 1: Evidence-based selection
Evidence-based selection is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Failure-mode awareness
Failure-mode awareness is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Implementation recommendation
Implementation recommendation is a core part of 'Momentum vs mean reversion strategy capstone'. Start with notation discipline: define time index, lag notation, and forecast horizon before estimating dependence. Then focus on temporal dependence structure and out-of-sample forecast discipline by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
**Plain-English interpretation:** Forecast error benchmark.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 2: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
**Plain-English interpretation:** Removes non-stationary level drift.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

### Formula 3: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
**Plain-English interpretation:** Lag-memory measurement.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Run the formula on rolling windows and inspect whether the value is stable across calm and stress periods.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $\phi$ | Autoregressive coefficient | dimensionless | 0.64 |
| $e_t$ | Forecast residual | return units | -0.003 |
| $\lambda$ | EWMA decay factor | 0 to 1 | 0.94 |

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Write a decision memo selecting a strategy for paper trading stage.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Capstone narrative and decision'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.
Common trap: Treating a non-stationary series as stationary and over-trusting one in-sample fit.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate final capstone report artifact with plots and metrics.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 12 Day 05): Explain RMSE and define every symbol clearly.
   - Model answer: "I use RMSE to convert raw prices into a decision-ready metric. The formula is $RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Capstone narrative and decision' matter in live trading systems?
   - Model answer: "Capstone narrative and decision matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Capstone narrative and decision as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## Reflection Question
What would invalidate your selected strategy next month?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 12 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [10]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1205)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


{'ar1_phi': -0.10249592813047463,
 'next_return_forecast': -0.001238733514514986,
 'acf_lag1': -0.10248220685064457,
 'acf_lag5': -0.0004519504034786784,
 'naive_rmse': 0.016055256056851906}

## Week 12 Day 05 Quiz

Topic: **Capstone narrative and decision**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [11]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 117.000
price_t = 118.228
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Capstone narrative and decision')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010496)
print('  gross_return_expected  =', 1.010496)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 117.0
  P_t    : 118.228
  r_t    : 0.010496 => 1.05%
  1+r_t  : 1.010496

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Capstone narrative and decision
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.010496
  gross_return_expected  = 1.010496


# Week 12 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 05: Capstone narrative and decision
- Previous lesson file: content/week-12/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 12 Day 07: Portfolio Project
- Next lesson file: content/week-12/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Re-run one sensitivity experiment
- Validate that costs are consistently applied
- Review capstone decision assumptions

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 12 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 12 Day 06: Revision Sprint
- Previous lesson file: content/week-12/day-06.md
- Today's deliverable: Capstone 3: Momentum vs mean reversion
- Next handoff target: Week 13 Day 01: Portfolio objective design
- Next lesson file: content/week-13/day-01.md

## Project Title
Capstone 3: Momentum vs mean reversion

## Problem Statement
Compare momentum and mean-reversion families under unified risk and cost rules.

## Data Sources
- Liquid ETF universe
- Daily prices
- Transaction-cost assumptions

## Implementation Steps
1. Design both signal families
2. Run equivalent backtests
3. Conduct robustness tests
4. Perform attribution analysis
5. Produce decision memo

## Evaluation Metrics
- Net return profile
- Sharpe proxy
- Drawdown
- Robustness score

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
